# 실험 01: RAG의 사전 준비 - Indexing (문서 색인화)

## 1. 실험 제목과 목표
- **제목**: Document Loader부터 Vector DB까지의 여정
- **목표**: RAG(검색 증강 생성)를 위해 질문이 들어오기 전, 미리 문서를 검색 가능한 숫자(벡터) 형태로 변환하여 저장하는 Indexing 파이프라인을 구축합니다.

## 2. 실험 개요
1. **실험 1**: 가상의 회사 규정 텍스트를 불러오고(Loader), 작은 덩어리로 자르고(Splitter), 벡터로 변환하여(Embedding) Vector DB에 저장합니다.
2. **실패/주의 케이스**: Chunk Size(자르는 크기)가 너무 작거나 Overlap(겹치는 부분)이 없을 때 문맥이 완전히 훼손되어 버리는 치명적 오류를 관찰합니다.

## 3. 라이브러리 import
> **참고**: `chromadb` 패키지가 설치되어 있어야 합니다. (`pip install chromadb langchain-chroma`)

In [1]:
import os
from dotenv import load_dotenv

from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma

## 4. 환경 변수 로드 및 임베딩 모델 준비

In [2]:
load_dotenv()
# 텍스트를 벡터로 바꿔줄 임베딩 모델입니다 (답변을 생성하는 LLM과는 다릅니다)
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

## 5. 실험 1: 정상적인 Indexing 파이프라인
교안의 [Loader -> Splitter -> Embedding -> Vector DB] 순서대로 진행합니다.

In [3]:
# 1. Document Loader 역할 (여기서는 편의상 리스트로 문서를 만듭니다)
docs = [
    Document(page_content="[제1장 총칙] 우리 회사의 연차 휴가는 1년에 15일 부여됩니다. 연차 신청은 최소 3일 전에 결재를 올려야 합니다."),
    Document(page_content="[제2장 복지] 재택근무는 주 2회 가능하며, 팀장의 사전 승인이 필수입니다. 식대는 매월 10만 원 지급됩니다.")
]

print("=== [1. 문서 분할 (Text Splitter)] ===")
# Chunk Size: 한 덩어리의 최대 길이
# Chunk Overlap: 문맥이 끊기지 않게 겹치게 할 길이
splitter = RecursiveCharacterTextSplitter(
    chunk_size=50,
    chunk_overlap=10
)

chunks = splitter.split_documents(docs)
print(f"원본 문서 개수: {len(docs)}개 -> 분할된 청크 개수: {len(chunks)}개\n")
for i, chunk in enumerate(chunks):
    print(f"청크 {i+1}: {chunk.page_content}")

print("\n=== [2. Vector DB 저장 (Embedding & Store)] ===")
# Chroma 메모리 DB에 청크들을 임베딩하여 저장합니다.
vectorstore = Chroma.from_documents(documents=chunks, embedding=embeddings)
print("-> 메모리 Vector DB 생성 완료! 이제 질문을 받아 검색할 준비가 되었습니다.")

=== [1. 문서 분할 (Text Splitter)] ===
원본 문서 개수: 2개 -> 분할된 청크 개수: 4개

청크 1: [제1장 총칙] 우리 회사의 연차 휴가는 1년에 15일 부여됩니다. 연차 신청은 최소 3일
청크 2: 신청은 최소 3일 전에 결재를 올려야 합니다.
청크 3: [제2장 복지] 재택근무는 주 2회 가능하며, 팀장의 사전 승인이 필수입니다. 식대는 매월
청크 4: 식대는 매월 10만 원 지급됩니다.

=== [2. Vector DB 저장 (Embedding & Store)] ===
-> 메모리 Vector DB 생성 완료! 이제 질문을 받아 검색할 준비가 되었습니다.


## 6. 실패/주의 케이스: 잘못된 Chunking 전략
Chunk Size를 무식하게 10으로 아주 작게 자르고, Overlap을 0으로 줘봅시다.

In [4]:
print("[잘못된 Splitter 설정 시뮬레이션]")
bad_splitter = RecursiveCharacterTextSplitter(chunk_size=10, chunk_overlap=0)
bad_chunks = bad_splitter.split_documents(docs)

print("잘못 분할된 청크 예시:")
for i in range(5):
    print(f"[{i+1}] {bad_chunks[i].page_content}")

print("\n-> 주의: [제1장 총칙], 우리 회사의, 연차 휴가는 ... 이렇게 문장이 다 토막납니다.")
print("   이 상태로 Vector DB에 들어가면, '연차 휴가 며칠이야?'라고 물어봤을 때 '연차 휴가는'이라는 청크만 찾아오고 '15일'이라는 정답이 있는 청크는 못 가져오는 대참사가 발생합니다.")
print("   따라서 교안에 명시된 것처럼 문맥이 유지되도록 Size와 Overlap을 넉넉히 주는 것이 RAG 성능의 80%를 좌우합니다.")

[잘못된 Splitter 설정 시뮬레이션]
잘못 분할된 청크 예시:
[1] [제1장 총칙]
[2] 우리 회사의 연차
[3] 휴가는 1년에
[4] 15일
[5] 부여됩니다. 연차

-> 주의: [제1장 총칙], 우리 회사의, 연차 휴가는 ... 이렇게 문장이 다 토막납니다.
   이 상태로 Vector DB에 들어가면, '연차 휴가 며칠이야?'라고 물어봤을 때 '연차 휴가는'이라는 청크만 찾아오고 '15일'이라는 정답이 있는 청크는 못 가져오는 대참사가 발생합니다.
   따라서 교안에 명시된 것처럼 문맥이 유지되도록 Size와 Overlap을 넉넉히 주는 것이 RAG 성능의 80%를 좌우합니다.


## 7. 결과 해석

1. **Indexing의 필수성**: LLM의 컨텍스트 윈도우(기억력 한계) 때문에 수천 장의 문서를 한 번에 프롬프트에 넣을 수 없습니다. 따라서 미리 잘게 쪼개어(Chunking) DB에 넣어둬야 합니다.
2. **임베딩(Embedding)**: 문장을 1536차원(OpenAI 기준)의 숫자 배열로 바꿉니다. 숫자로 바뀌어야 컴퓨터가 의미의 '유사도'를 수학적으로 계산할 수 있습니다.
3. **Vector DB**: 관계형 DB(MySQL)나 NoSQL(MongoDB)이 아닌, 텍스트의 '의미(벡터)'를 저장하고 검색하는 데 특화된 데이터베이스(Chroma, FAISS 등)입니다.

## 8. Notion 실험 로그

### 발견한 문제점 (또는 차이점)
* `RecursiveCharacterTextSplitter`를 쓰면 기본적으로 문단(\n\n) -> 문장(\n) -> 단어(공백) 순서로 똑똑하게 잘라준다는 것을 배움.
* 하지만 개발자가 `chunk_size`를 너무 빡빡하게 잡으면 아무리 똑똑한 Splitter라도 문맥을 다 부숴버림을 확인.

### 다음 개선 방향
* DB에 저장은 잘 끝났음.
* 이제 사용자가 질문을 던졌을 때, 이 Vector DB에서 가장 질문과 의미가 비슷한 청크(문서 조각)를 찾아오는 **Retriever(검색기)**를 만들어야 함.